# Additional SDC Schema Elements and Attributes

This notebook documents additional SDC schema elements and attributes that are defined in the SDC schema files but are not yet fully documented in the main SDC Schema documentation. These elements and attributes extend the core functionality described in the main schema documentation.

**Note:** This document is based on a comparison between the SDC schema XSD files and the documented elements in `Schema/SDCSchema.ipynb`. Some of these elements may be:
- Advanced features for specific use cases
- Part of package management functionality
- Internal implementation details
- Rarely used or deprecated features

This document focuses on the most commonly used and important missing elements and attributes that SDC implementers should be aware of.


## Events

SDC supports a comprehensive event system that allows forms to respond to various user interactions and lifecycle events. Events can trigger actions, guards, and other form behaviors.

### Form Lifecycle Events

These events are fired at different stages of the form lifecycle:

* `BeforeLoadForm`: This event is fired before the page is loaded into memory, and before stored form data is loaded. It may be used for authentication, to retrieve/prepare stored data, and/or to control form rendering according to user preferences.

* `BeforeLoadData`: This event is fired after the page is loaded into memory, before stored form data is loaded, and before the form is visible. It may be used to determine the data to be loaded and to perform the data loading.

* `BeforeShowForm`: This event is fired after the page is loaded into memory, after the data is loaded into the form, but before the form is displayed. It may be used to perform form activities that are controlled by the loaded data.

* `BeforeDataSubmit`: This event is fired before form data is submitted to a Form Receiver. It may be used for validation, data transformation, or other pre-submission tasks.

* `BeforeCloseForm`: This event is fired before the form is closed. It may be used for cleanup, final validation, or saving intermediate data.

### User Interaction Events

These events are fired in response to user interactions with form elements:

* `OnEvent`: A generic event handler that can be customized with an `eventName` attribute. This allows form designers to define custom events for specific use cases.

* `OnClick`: Fired when a user clicks on an element (typically used with ButtonAction elements).

* `OnEnter`: Fired when a user enters (focuses) a form element.

* `OnExit`: Fired when a user exits (loses focus) a form element.

* `OnSelect`: Fired when a ListItem is selected by the user.

* `OnDeselect`: Fired when a ListItem is deselected by the user.

* `AfterChange`: Fired after a value in a form element has been changed.

### Example: Using Events

```xml
<FormDesign>
    <BeforeLoadForm>
        <!-- Actions to perform before form loads -->
    </BeforeLoadForm>
    <BeforeShowForm>
        <!-- Actions to perform before form displays -->
    </BeforeShowForm>
    <Question ID="Q1" title="Enter a value">
        <OnEnter>
            <!-- Actions when user enters this question -->
        </OnEnter>
        <AfterChange>
            <!-- Actions after value changes -->
        </AfterChange>
    </Question>
</FormDesign>
```


## Actions and Guards

Actions and Guards provide the mechanism for implementing form logic and conditional behavior in SDC forms.

### Guards

Guards are conditional elements that control the activation, selection, or visibility of form elements:

* `ActivateIf`: A guard that activates (enables and makes visible) form elements when its condition is true. When the condition becomes false, the elements are deactivated.

* `DeActivateIf`: A guard that deactivates (disables and/or hides) form elements when its condition is true. When the condition becomes false, the elements are reactivated.

* `SelectIf`: A guard that automatically selects a ListItem when its condition is true. When the condition becomes false, the ListItem is deselected.

* `DeselectIf`: A guard that automatically deselects a ListItem when its condition is true. When the condition becomes false, the ListItem may be selected again.

### Actions

Actions are elements that perform specific operations when triggered by events:

* `Action` / `Actions`: Container elements for one or more action elements. Actions can be grouped together to execute multiple operations in sequence.

* `SetValue`: Sets the value of a form element (typically a ResponseField or ListItemResponseField).

* `SetAttributeValue`: Sets the value of an attribute on a form element.

* `CallFunction`: Calls a function (typically a web service or JavaScript function) and optionally uses the return value.

* `CallBoolFunction`: Calls a boolean function and uses the return value to control form behavior.

### Example: Using Guards and Actions

```xml
<Question ID="Q1" title="Do you have allergies?">
    <ListField>
        <List>
            <ListItem ID="LI1" title="Yes"/>
            <ListItem ID="LI2" title="No"/>
        </List>
    </ListField>
</Question>

<Question ID="Q2" title="Describe your allergies">
    <ActivateIf>
        <!-- Activate Q2 only if LI1 (Yes) is selected in Q1 -->
        <hasSelectionsExact itemNames="Q1" selectedItemNames="LI1"/>
    </ActivateIf>
    <ResponseField>
        <Response>
            <string/>
        </Response>
    </ResponseField>
</Question>
```


## Additional FormDesign Attributes

Beyond the attributes documented in the main schema documentation, `FormDesign` supports additional attributes for versioning and instance management:

### Version Management Attributes

* `versionPrev`: Identifies the immediate previous version of the current FDF. The format is the same as `version`. The primary role of this optional attribute is to allow automated comparisons between a current FDF and the immediate previous FDF version. This is often helpful when deciding whether to adopt a newer version of an FDF.

### Instance Management Attributes

These attributes are used in FDF-R documents (forms with user responses) to track form instances:

* `instanceID`: A unique string (e.g., a GUID) used to identify a unique instance of a form, such as a form used during a single patient encounter. The `instanceID` is used to track saved form responses across time and across multiple episodes of editing by end-users. This string does not change for each edit session of a form or package instance. The `instanceID` is required in an FDF-R; it is not allowed in an FDF.

* `instanceVersion`: A timestamp (xs:dateTime) used to identify a unique instance of a form. Used for tracking form responses across time and across multiple episodes of editing by end-users. This field must change for each edit session of a form instance. The `instanceVersion` is required in an FDF-R; it is not allowed in an FDF.

* `instanceVersionPrev`: A unique dateTime used to identify the immediate previous instance of a form instance. Used for tracking form responses across time and across multiple episodes of editing by end-users. This field must change for each edit session of a form instance.

### Example: FormDesign with Instance Attributes

```xml
<FormDesign 
    baseURI="cap.org"
    lineage="Lung.Bmk.227"
    version="1.001.011.RC1"
    ID="Lung.Bmk.227_1.001.011.RC1_sdcFDF"
    fullURI="_baseURI=cap.org&amp;_lineage=Lung.Bmk.227&amp;_version=1.001.011.RC1&amp;_docType=sdcFDF"
    instanceID="Abc1dee2fg987"
    instanceVersion="2019-07-16T19:20:30+01:00"
    instanceVersionPrev="2019-07-15T14:30:00+01:00">
    <!-- Form content -->
</FormDesign>
```


## ListField and ListItem Attributes

### ListField Attributes

* `colTextDelimiter`: Specifies the delimiter used to separate column text in multi-column lists. This is particularly useful when using `ListHeaderText` with multiple columns or when parsing data from `LookupEndPoint` responses.

* `minSelections`: Specifies the minimum number of ListItems that must be selected in a multi-select question. This attribute works in conjunction with `maxSelections` to define selection constraints.

* `ordered`: A boolean attribute that indicates whether the list items should be displayed in a specific order. When `ordered="true"`, the order of ListItems in the XML should be preserved in the DEF.

* `sorted`: A boolean attribute that indicates whether the list items should be sorted. When `sorted="true"`, the ListItems may be automatically sorted by the Form Filler.

* `sortDirection`: When `sorted="true"`, this attribute specifies the sort direction. Values can be "ascending" or "descending".

### ListItem Attributes

* `selectionActivatesItems`: Selecting the current ListItem will enable the named items specified in this attribute's content. The attribute value contains a space-separated list of item names (IDs). Prefixing any name with a hyphen (-) will reverse the behavior (i.e., the named items will be disabled). Unselecting the ListItem will reverse this behavior. Prefixing the name with a tilde (~) will suppress this reversal behavior.

* `selectionSelectsListItems`: Selecting the current ListItem will select the named ListItems specified in this attribute's content. The attribute value contains a space-separated list of ListItem names (IDs). Prefixing any name with a hyphen (-) will reverse the behavior (i.e., the named ListItems will be deselected). Unselecting the ListItem will reverse this behavior. Prefixing the name with a tilde (~) will suppress this reversal behavior.

* `associatedValue`: A typed value (e.g., an integer) that is uniquely associated with a ListItem. An example is the integer 10 for a ListItem with title that reads "10 o'clock". Typically these values are set to be used in calculations or other algorithms. In general, they can be treated something like a user-entered response on the `ListItemResponseField` of a selected ListItem. This field should not be used for terminologies or local codes. The `CodedValue` type should be used for these kinds of metadata.

* `associatedValueType`: The data type of `associatedValue`. Default is string. Valid values include: string, integer, decimal, date, dateTime, etc.

### Example: Using ListItem Attributes

```xml
<Question ID="Q1" title="Select your options">
    <ListField maxSelections="0" ordered="true" sorted="true" sortDirection="ascending">
        <List>
            <ListItem ID="LI1" title="Option 1" 
                associatedValue="1" 
                associatedValueType="integer"
                selectionActivatesItems="Q2 Q3">
                <!-- Selecting LI1 activates Q2 and Q3 -->
            </ListItem>
            <ListItem ID="LI2" title="Option 2" 
                associatedValue="2" 
                associatedValueType="integer"
                selectionSelectsListItems="LI3">
                <!-- Selecting LI2 automatically selects LI3 -->
            </ListItem>
            <ListItem ID="LI3" title="Option 3"/>
        </List>
    </ListField>
</Question>
```


## Response Field Attributes

### ResponseField and ListItemResponseField Attributes

* `responseRequired`: A boolean attribute that indicates whether a response must be entered in the ResponseField or ListItemResponseField. When `responseRequired="true"`, the Form Filler should validate that a value has been entered before allowing form submission. This attribute is particularly useful for `ListItemResponseField` elements where a response is required only when the parent ListItem is selected.

* `defaultListItemDataType`: Specifies the default data type for ListItem responses when the data type is not explicitly specified. This attribute is typically used on the parent Question or ListField element.

### Example: Required Response Field

```xml
<Question ID="Q1" title="Select an option">
    <ListField>
        <List>
            <ListItem ID="LI1" title="Other (specify)">
                <ListItemResponseField responseRequired="true">
                    <Response>
                        <string maxLength="100"/>
                    </Response>
                </ListItemResponseField>
            </ListItem>
        </List>
    </ListField>
</Question>
```


## Data Type Validation Attributes

SDC supports comprehensive data type validation through various attributes on datatype elements. These attributes provide fine-grained control over the acceptable values for user responses.

### Numeric Validation Attributes

* `allowGT`, `allowGTE`, `allowLT`, `allowLTE`: Boolean attributes that control whether values greater than (GT), greater than or equal to (GTE), less than (LT), or less than or equal to (LTE) the specified bounds are allowed. These work in conjunction with `maxInclusive`, `minInclusive`, `maxExclusive`, and `minExclusive` attributes.

* `allowAPPROX`: A boolean attribute that allows approximate matching for numeric values. This is useful when exact matches are not required.

* `allowNull` / `allowNulls`: Boolean attributes that control whether null or empty values are allowed for the response.

* `fractionDigits`: Specifies the maximum number of decimal places allowed for decimal and float datatypes.

* `totalDigits`: Specifies the total number of digits (including decimal places) allowed for numeric datatypes.

* `maxExclusive` / `minExclusive`: Specifies exclusive bounds for numeric values. Values equal to these bounds are not allowed.

* `maxInclusive` / `minInclusive`: Specifies inclusive bounds for numeric values. Values equal to these bounds are allowed.

### String Validation Attributes

* `maxLength` / `minLength`: Specifies the maximum and minimum length for string values. These are commonly used and are mentioned in the main documentation.

* `pattern`: Specifies a regular expression pattern that the string value must match.

* `mask`: Specifies an input mask that defines the format for user input (e.g., phone number format, date format).

### Example: Comprehensive Data Type Validation

```xml
<Question ID="Q1" title="Enter a percentage (1-100)">
    <ResponseField>
        <Response>
            <integer 
                minInclusive="1" 
                maxInclusive="100"
                allowGT="false"
                allowLT="false"
                allowNull="false"/>
        </Response>
        <TextAfterResponse val="%"/>
    </ResponseField>
</Question>

<Question ID="Q2" title="Enter a decimal value">
    <ResponseField>
        <Response>
            <decimal 
                minInclusive="0.0"
                maxInclusive="1000.0"
                fractionDigits="2"
                totalDigits="6"
                allowAPPROX="true"/>
        </Response>
    </ResponseField>
</Question>

<Question ID="Q3" title="Enter a phone number">
    <ResponseField>
        <Response>
            <string 
                pattern="^\d{3}-\d{3}-\d{4}$"
                mask="###-###-####"
                maxLength="12"/>
        </Response>
    </ResponseField>
</Question>
```


## List Features

### ListHeaderText

The `ListHeaderText` element provides a header row for a set of list items. If the list has more than one column, the column text is separated by the `colTextDelimiter` attribute on the parent `ListField` element.

### LookupEndPoint

The `LookupEndPoint` element represents list items that are derived from a web service call, instead of an explicit set of `ListItem` nodes specified in the FormDesign XML. The endpoint must return a list separated into individual list items by the `colTextDelimiter` value specified in the parent `ListField`.

### Illegal Co-Selection Rules

* `IllegalCoSelectedListItems`: Specifies ListItems that cannot be selected together. This is useful for enforcing mutual exclusivity rules beyond what can be achieved with single-select questions.

* `IllegalListItemPairings`: Specifies pairs of ListItems that cannot be selected together. This provides more granular control than `IllegalCoSelectedListItems`.

### Example: Using List Features

```xml
<Question ID="Q1" title="Select from dynamic list">
    <ListField colTextDelimiter="|" maxSelections="1">
        <ListHeaderText>Name|Code|Description</ListHeaderText>
        <LookupEndPoint 
            serverURI="https://example.com/api/lookup"
            functionName="getListItems"
            includesHeaderRow="true">
            <!-- Parameters for the lookup -->
        </LookupEndPoint>
    </ListField>
</Question>

<Question ID="Q2" title="Select options (with restrictions)">
    <ListField maxSelections="0">
        <List>
            <ListItem ID="LI1" title="Option 1"/>
            <ListItem ID="LI2" title="Option 2"/>
            <ListItem ID="LI3" title="Option 3"/>
        </List>
        <IllegalCoSelectedListItems>LI1 LI2</IllegalCoSelectedListItems>
        <!-- LI1 and LI2 cannot be selected together -->
    </ListField>
</Question>
```


## Coding and Terminology

While `CodedValue` is mentioned in the main documentation, the following elements provide additional functionality for coding and terminology:

### CodedValue Child Elements

* `Code`: A standard code, or a local value from a custom coding system, that can be used to consistently identify, or provide a standard value for, the coded item.

* `CodeText`: The human-readable text that accompanies the assigned code and represents the code's precise meaning (semantics) or usage.

* `CodeURI`: Web resource that provides information about the code.

* `CodeSystem`: The parent element whose children define the system that creates and maintains the standards for the code map.

### CodeSystem Child Elements

* `CodeSystemName`: The name of the coding system, as recommended by the coding system curators, or as recommended by the agency that creates standards for the code map in use.

* `CodeSystemURI`: Web resource that uniquely identifies the coding system.

* `OID`: The ISO object identifier (OID) for the coding system, as found at the HL7 OID Registry: https://www.hl7.org/oid/index.cfm

* `Version`: Version of the coding system, using the version format defined by the coding system.

* `ReleaseDate`: The day that the selected version of the coding system was released for general use by the coding system curators.

### Example: Comprehensive CodedValue

```xml
<Question ID="Q1" title="Diagnosis">
    <ListField>
        <List>
            <ListItem ID="LI1" title="Malignant neoplasm">
                <CodedValue>
                    <Code>C80.1</Code>
                    <CodeText>Malignant (primary) neoplasm, unspecified</CodeText>
                    <CodeURI>https://www.icd10data.com/ICD10CM/Codes/C00-D49/C76-C80/C80-/C80.1</CodeURI>
                    <CodeSystem>
                        <CodeSystemName>ICD-10-CM</CodeSystemName>
                        <CodeSystemURI>https://www.cdc.gov/nchs/icd/icd10cm.htm</CodeSystemURI>
                        <OID>2.16.840.1.113883.6.90</OID>
                        <Version>2023</Version>
                        <ReleaseDate>2022-10-01</ReleaseDate>
                    </CodeSystem>
                </CodedValue>
            </ListItem>
        </List>
    </ListField>
</Question>
```


## InjectForm

The `InjectForm` element allows a form or portion of a form to be imported into the current form at a specific location. It enables the composition of forms from other forms or parts of other forms.

### InjectForm Attributes

* `InjectionSourceURI`: Required attribute that specifies the source of the SDC FormDesign, Section, or Question to inject. The suggested form of the URI is:
  - `serverURI + \fullURI` for the FDF (retrieves the latest package version with FDF responses)
  - `serverURI + \instanceVersionURI` for the FDF-R (retrieves a specific package version with FDF responses)

* `rootItemID`: Required attribute that specifies the ID of the form or form part that will be injected. It must point to a valid FormDesign, Section, or Question element.

* `serverURI`: Optional attribute that specifies the server from which the injected package will be retrieved.

### InjectForm Content

The `InjectForm` element can contain:
- A `Section` element (the injected section)
- A `Question` element (the injected question)
- A `FormDesign` element (the injected form)

In practice, using an injected section requires some or all of the injected FormDesign XML to be injected under the `InjectForm` element. However, in a "raw" form (not yet filled out), the FormDesign element would generally be empty; only the top-level `InjectFormType` attributes would be used to point to the parts to be later injected.

### Example: Using InjectForm

```xml
<Section ID="S1" title="Main Section">
    <ChildItems>
        <Question ID="Q1" title="Question 1"/>
        
        <!-- Inject a section from another form -->
        <InjectForm 
            InjectionSourceURI="https://example.com/forms/template.xml"
            rootItemID="S_CommonData"
            serverURI="https://example.com/">
            <!-- Injected content will be inserted here at runtime -->
        </InjectForm>
        
        <Question ID="Q2" title="Question 2"/>
    </ChildItems>
</Section>
```


## Display and Behavior Attributes

These attributes control the display and interactive behavior of form elements:

### Visibility and State Attributes

* `visible`: Boolean attribute that controls whether an element is visible in the DEF. When `visible="false"`, the element is hidden but may still be present in the form structure.

* `enabled`: Boolean attribute that controls whether an element can be interacted with. When `enabled="false"`, the element is disabled (grayed out) and cannot receive user input.

* `readOnly`: Boolean attribute that makes an element read-only. When `readOnly="true"`, the element displays its value but cannot be edited by the user. This is mentioned in the main documentation but not fully explained.

* `isActive`: Boolean attribute that indicates whether an element is currently active. An active element is both visible and enabled.

* `isEnabled`: Boolean attribute that indicates whether an element is enabled.

* `isReadOnly`: Boolean attribute that indicates whether an element is read-only.

* `isRequired`: Boolean attribute that indicates whether an element is required.

* `isSelected`: Boolean attribute that indicates whether a ListItem is selected.

* `isVisible`: Boolean attribute that indicates whether an element is visible.

### Reporting Attributes

* `showInReport`: Boolean attribute that controls whether an element should appear in generated reports. This works in conjunction with the `reportText` Property to control report output.

* `displayState`: Attribute that specifies the display state of an element. Values may include: "expanded", "collapsed", "hidden", etc.

### Example: Using Display Attributes

```xml
<Question ID="Q1" title="Read-only question" readOnly="true">
    <ListField>
        <List>
            <ListItem ID="LI1" title="Pre-selected value" selected="true"/>
        </List>
    </ListField>
</Question>

<Question ID="Q2" title="Hidden question" visible="false">
    <ResponseField>
        <Response>
            <string/>
        </Response>
    </ResponseField>
</Question>

<Question ID="Q3" title="Disabled question" enabled="false">
    <ResponseField>
        <Response>
            <string/>
        </Response>
    </ResponseField>
</Question>
```


## Additional Elements

### Header and Footer

While mentioned in the main documentation, `Header` and `Footer` are Section elements that provide persistent header and footer content for forms:

* `Header`: An optional Section that stays at the top of a form. Content in the Header typically appears on every page or screen of the form.

* `Footer`: An optional Section that stays at the bottom of a form. Content in the Footer typically appears on every page or screen of the form.

### Rules

The `Rules` element contains form logic, validation rules, and business rules that govern form behavior. Rules can include:
- Conditional logic
- Validation constraints
- Calculation rules
- Cross-field validation

### ResponseValue

The `ResponseValue` element represents the user's response to a lookup list when using `LookupEndPoint`. The response is recorded as a coding, terminology, classification, keyword, or local value. Multiple selections from the lookup list may be allowed.

### TypedValue

While `TypedValue` is mentioned in the context of Properties, it can also be used with other elements to provide strongly-typed values. `TypedValue` allows you to specify the exact data type for a value, ensuring proper validation and type checking.

### Example: Using Header, Footer, and Rules

```xml
<FormDesign>
    <Header>
        <Section ID="Header_Section" title="">
            <ChildItems>
                <DisplayedItem ID="DI_Header" title="Form Header Information"/>
            </ChildItems>
        </Section>
    </Header>
    
    <Body>
        <!-- Main form content -->
    </Body>
    
    <Footer>
        <Section ID="Footer_Section" title="">
            <ChildItems>
                <DisplayedItem ID="DI_Footer" title="Form Footer Information"/>
            </ChildItems>
        </Section>
    </Footer>
    
    <Rules>
        <!-- Form rules and logic -->
    </Rules>
</FormDesign>
```


## Summary

This notebook has documented additional SDC schema elements and attributes that extend the core functionality described in the main schema documentation. Key areas covered include:

1. **Events**: Form lifecycle and user interaction events
2. **Actions and Guards**: Conditional logic and form behavior control
3. **Additional FormDesign Attributes**: Version and instance management
4. **ListField and ListItem Attributes**: Advanced list functionality
5. **Response Field Attributes**: Response validation and requirements
6. **Data Type Validation Attributes**: Comprehensive validation controls
7. **List Features**: Dynamic lists and selection rules
8. **Links and Binary Content**: Rich media support
9. **Coding and Terminology**: Extended terminology support
10. **InjectForm**: Form composition and reuse
11. **Display and Behavior Attributes**: UI control attributes
12. **Additional Elements**: Header, Footer, Rules, and more

These elements and attributes provide SDC implementers with powerful tools for creating sophisticated, interactive forms with complex validation, conditional logic, and rich content support.

For more information on specific elements, refer to the SDC schema XSD files in the `SDC-Schema-Packages` folder, or consult the IHE SDC Profile documentation.
